In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
# Load the cleaned dataset
df = pd.read_csv("heart_disease_cleaned.csv")
print(f"Shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['HeartDiseaseorAttack'].value_counts())
print(f"\nPositive class rate: {df['HeartDiseaseorAttack'].mean():.2%}")
print(f"Features: {df.shape[1] - 1}")
print(f"Samples: {df.shape[0]}")

# Separate features and target
X = df.drop(columns=['HeartDiseaseorAttack'])
y = df['HeartDiseaseorAttack']

Shape: (229781, 22)

Target distribution:
HeartDiseaseorAttack
0.0    206064
1.0     23717
Name: count, dtype: int64

Positive class rate: 10.32%
Features: 21
Samples: 229781


In [3]:
# Train / Validation / Test Split

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Second split: split the 30% into 15/15
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print("Split sizes:")
print(f"  Train: {X_train.shape[0]:,} ({X_train.shape[0]/len(df):.1%})")
print(f"  Val:   {X_val.shape[0]:,} ({X_val.shape[0]/len(df):.1%})")
print(f"  Test:  {X_test.shape[0]:,} ({X_test.shape[0]/len(df):.1%})")
print(f"\nPositive class rate per split:")
print(f"  Train: {y_train.mean():.2%}")
print(f"  Val:   {y_val.mean():.2%}")
print(f"  Test:  {y_test.mean():.2%}")


Split sizes:
  Train: 160,846 (70.0%)
  Val:   34,467 (15.0%)
  Test:  34,468 (15.0%)

Positive class rate per split:
  Train: 10.32%
  Val:   10.32%
  Test:  10.32%


In [4]:
# Feature Scaling
scaler = StandardScaler()

# Fit on train, transform all three
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val), columns=X_val.columns, index=X_val.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

print(f"Train mean (should be ~0): {X_train_scaled.mean().mean():.6f}")
print(f"Train std  (should be ~1): {X_train_scaled.std().mean():.6f}")
print(f"Val mean   (should be near 0): {X_val_scaled.mean().mean():.6f}")

Train mean (should be ~0): 0.000000
Train std  (should be ~1): 1.000003
Val mean   (should be near 0): -0.000832


In [5]:
# SMOTE (Training Set Only)
smote = SMOTE(random_state=RANDOM_STATE)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:")
print(f"  Train size: {X_train_scaled.shape[0]:,}")
print(f"  Class 0: {(y_train == 0).sum():,}  |  Class 1: {(y_train == 1).sum():,}")
print(f"\nAfter SMOTE:")
print(f"  Train size: {X_train_resampled.shape[0]:,}")
print(f"  Class 0: {(y_train_resampled == 0).sum():,}  |  Class 1: {(y_train_resampled == 1).sum():,}")
print(f"  Balance ratio: {(y_train_resampled == 0).sum() / (y_train_resampled == 1).sum():.2f}:1")

Before SMOTE:
  Train size: 160,846
  Class 0: 144,244  |  Class 1: 16,602

After SMOTE:
  Train size: 288,488
  Class 0: 144,244  |  Class 1: 144,244
  Balance ratio: 1.00:1


In [6]:
import os
os.makedirs("Data", exist_ok=True)

# SMOTE'd training data (for models that use oversampled training)
X_train_resampled.to_csv("Data/X_train_resampled.csv", index=False)
y_train_resampled.to_csv("Data/y_train_resampled.csv", index=False)

# Original scaled training data (for models like XGBoost that use scale_pos_weight instead of SMOTE)
X_train_scaled.to_csv("Data/X_train_scaled.csv", index=False)
y_train.to_csv("Data/y_train.csv", index=False)

# Validation set
X_val_scaled.to_csv("Data/X_val_scaled.csv", index=False)
y_val.to_csv("Data/y_val.csv", index=False)

# Test set (held out until final evaluation)
X_test_scaled.to_csv("Data/X_test_scaled.csv", index=False)
y_test.to_csv("Data/y_test.csv", index=False)

print("Exported splits to Data/:")
for f in sorted(os.listdir("Data")):
    size = pd.read_csv(f"Data/{f}").shape
    print(f"  {f:30s} {size}")

print("\nDone. All downstream notebooks can load from Data/.")

Exported splits to Data/:
  X_test_scaled.csv              (34468, 21)
  X_train_resampled.csv          (288488, 21)
  X_train_scaled.csv             (160846, 21)
  X_val_scaled.csv               (34467, 21)
  y_test.csv                     (34468, 1)
  y_train.csv                    (160846, 1)
  y_train_resampled.csv          (288488, 1)
  y_val.csv                      (34467, 1)

Done. All downstream notebooks can load from Data/.
